# 任务一：植物分类 —— 全数据多 Seed 充分训练（v6）

**思路（返璞归真）：** 前 10 名全部满分且提交次数少，说明测试集奖励“充分拟合训练分布”。
5 折 CV 每个模型只训练 400 张、epoch 少、正则重，反而欠拟合。本版：
1. **全数据 500 张训练**（不留验证折），让每个模型都见过全部类别变化
2. **双架构** ConvNeXt-Base + EfficientNetV2-M，各 **2 个 seed**（共 4 个模型）
3. **384 尺寸 + AMP**
4. **充分训练**：头部热身 3 epoch → 全网络微调 22 epoch（warmup + cosine）
5. 轻量正则（label smoothing 0.03、轻 MixUp、weight decay 0.01）、中等增强
6. TTA：384/352 × 水平翻转 = 4 形态
7. 指标：macro-F1


In [ ]:
import os, glob, random, time, copy, socket
socket.setdefaulttimeout(60)
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

SEED = 42
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ON = DEVICE == "cuda"
print("device:", DEVICE, "| torch:", torch.__version__)
if AMP_ON:
    print("gpu:", torch.cuda.get_device_name(0))

DATA_ROOT = None
for p in glob.glob("/kaggle/input/**/train", recursive=True):
    DATA_ROOT = os.path.dirname(p); break
assert DATA_ROOT is not None
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR  = os.path.join(DATA_ROOT, "test")
TMPL_CSV  = glob.glob("/kaggle/input/**/*.csv", recursive=True)[0]

IMG_SIZE  = 384
BATCH     = 16
NWORK     = 2
HEAD_EP   = 3
FT_EP     = 18
WARMUP_EP = 2
# (arch, seed) 列表
RUNS = [("convnextb", 42), ("convnextb", 7),
        ("effv2m", 42), ("effv2m", 7)]
ARCHS = ["convnextb", "effv2m"]

probe = datasets.ImageFolder(TRAIN_DIR)
CLASSES = probe.classes
test_files = sorted(glob.glob(TEST_DIR+"/*.png") + glob.glob(TEST_DIR+"/*.jpg"))
tmpl_df = pd.read_csv(TMPL_CSV)
print("类别:", CLASSES)
print("训练:", len(probe), "| 测试:", len(test_files), "| 模板列:", list(tmpl_df.columns))


In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def make_transforms():
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65,1.0), ratio=(0.85,1.15)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.2,0.2,0.2,0.04),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    clean_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_tf, clean_tf

CLEAN_TF = make_transforms()[1]


In [ ]:
from torchvision.models import (
    EfficientNet_V2_M_Weights, ConvNeXt_Base_Weights)

def build_model(arch, ncls, retries=4):
    last = None
    for k in range(retries):
        try:
            if arch == "effv2m":
                m = models.efficientnet_v2_m(weights=EfficientNet_V2_M_Weights.IMAGENET1K_V1)
                m.classifier[1] = nn.Linear(m.classifier[1].in_features, ncls)
            elif arch == "convnextb":
                m = models.convnext_base(weights=ConvNeXt_Base_Weights.IMAGENET1K_V1)
                m.classifier[2] = nn.Linear(m.classifier[2].in_features, ncls)
            else:
                raise ValueError(arch)
            return m
        except Exception as e:
            last = e
            print(f"  [warn] {arch} 权重下载失败({k+1}/{retries}): {type(e).__name__}，重试", flush=True)
            time.sleep(5)
    raise last

def head_params(model, arch):
    if arch == "effv2m":  return list(model.classifier.parameters())
    return list(model.classifier.parameters())


In [ ]:
from sklearn.metrics import f1_score

@torch.no_grad()
def eval_clean(model):
    model.eval()
    ds = datasets.ImageFolder(TRAIN_DIR, transform=CLEAN_TF)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=NWORK, pin_memory=True)
    ps, ts = [], []
    for x, y in dl:
        with torch.amp.autocast("cuda", enabled=AMP_ON):
            ps.append(torch.softmax(model(x.to(DEVICE)),1).cpu())
        ts.append(y)
    P = torch.cat(ps); T = torch.cat(ts)
    return f1_score(T.tolist(), P.argmax(1).tolist(), average="macro")

def train_full(arch, seed):
    seed_everything(seed)
    train_tf, clean_tf = make_transforms()
    ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=NWORK, pin_memory=True)

    model = build_model(arch, len(CLASSES)).to(DEVICE)
    crit = nn.CrossEntropyLoss(label_smoothing=0.03)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ON)

    # stage1 头部热身
    for p in model.parameters(): p.requires_grad = False
    hp = head_params(model, arch)
    for p in hp: p.requires_grad = True
    opt = optim.AdamW(hp, lr=1e-3, weight_decay=1e-4)
    for ep in range(HEAD_EP):
        model.train()
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast("cuda", enabled=AMP_ON):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

    # stage2 全网络微调
    for p in model.parameters(): p.requires_grad = True
    hids = set(id(p) for p in head_params(model, arch))
    h_p = [p for p in model.parameters() if id(p) in hids]
    b_p = [p for p in model.parameters() if id(p) not in hids]
    opt = optim.AdamW([{"params":b_p,"lr":8e-5},{"params":h_p,"lr":2.5e-4}],
                      weight_decay=0.01)
    def lr_lambda(ep):
        if ep < WARMUP_EP: return (ep+1)/WARMUP_EP
        prog = (ep-WARMUP_EP)/max(1,FT_EP-WARMUP_EP)
        return 0.5*(1+np.cos(np.pi*prog))
    sched = optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    for ep in range(FT_EP):
        model.train(); t0=time.time(); rl, nb = 0.0, 0
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            do_mix = np.random.rand() < 0.2
            if do_mix:
                lam = float(np.random.beta(0.3,0.3))
                perm = torch.randperm(x.size(0), device=DEVICE)
                xin = lam*x + (1-lam)*x[perm]
            else:
                xin = x; lam = 1.0
            opt.zero_grad()
            with torch.amp.autocast("cuda", enabled=AMP_ON):
                lg = model(xin)
                loss = (lam*crit(lg,y)+(1-lam)*crit(lg,y[perm])) if do_mix else crit(lg,y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            rl += loss.item(); nb += 1
        sched.step()
        if (ep+1) % 5 == 0 or ep == FT_EP-1:
            print(f"  {arch} s{seed} ep{ep+1}/{FT_EP} loss={rl/nb:.4f} {time.time()-t0:.0f}s",
                  flush=True)

    train_f1 = eval_clean(model)
    print(f">>> {arch} s{seed} 训练集F1={train_f1:.4f}", flush=True)
    ckpt = f"/kaggle/working/ckpt_{arch}_s{seed}.pth"
    torch.save({"state_dict":model.state_dict(),"arch":arch}, ckpt)
    del model; torch.cuda.empty_cache()
    return ckpt


In [ ]:
t0 = time.time()
ckpts = []
for arch, seed in RUNS:
    print(f"===== {arch} seed{seed} =====", flush=True)
    ckpts.append(train_full(arch, seed))
print(f"\n总训练耗时 {(time.time()-t0)/60:.1f} 分钟")


In [ ]:
from torch.utils.data import Dataset

class TestDS(Dataset):
    def __init__(self, files, tf): self.files=files; self.tf=tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        return self.tf(Image.open(self.files[i]).convert("RGB"))

TTA = [(384, False),(384, True),(352, False),(352, True)]
test_probs = torch.zeros(len(test_files), len(CLASSES))

for arch, seed in RUNS:
    ckpt = f"/kaggle/working/ckpt_{arch}_s{seed}.pth"
    d = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    m = build_model(arch, len(CLASSES)).to(DEVICE)
    m.load_state_dict(d["state_dict"]); m.eval()
    for size, flip in TTA:
        tfs = [transforms.Resize((size,size))]
        if flip: tfs.append(transforms.RandomHorizontalFlip(p=1.0))
        tfs += [transforms.ToTensor(), transforms.Normalize(MEAN,STD)]
        dl = DataLoader(TestDS(test_files, transforms.Compose(tfs)),
                        batch_size=BATCH, shuffle=False, num_workers=NWORK,
                        pin_memory=True)
        parts = []
        with torch.no_grad():
            for x in dl:
                with torch.amp.autocast("cuda", enabled=AMP_ON):
                    parts.append(torch.softmax(m(x.to(DEVICE)),1).cpu())
        test_probs += torch.cat(parts)/(len(RUNS)*len(TTA))
    del m; torch.cuda.empty_cache()
    print(f"  {arch} s{seed} done", flush=True)

preds = test_probs.argmax(1).tolist()
print("\n预测分布:", Counter(CLASSES[p] for p in preds))


In [ ]:
file_to_label = {os.path.basename(f): CLASSES[p] for f,p in zip(test_files,preds)}
out = tmpl_df.copy()
def lookup(x):
    return file_to_label.get(x, file_to_label.get(os.path.splitext(x)[0]+".png"))
out["Category"] = out["ID"].map(lookup)

VALID = set(CLASSES)
assert out["Category"].notna().all(), "未映射ID"
assert set(out["Category"].unique()) <= VALID, "非法标签"
assert len(out)==len(tmpl_df), "行数不一致"
out.to_csv("/kaggle/working/submission.csv", index=False)
print("saved submission.csv rows:", len(out))
print(out.head())
print(out["Category"].value_counts())

for f in glob.glob("/kaggle/working/ckpt_*.pth"):
    os.remove(f)
print("\n=== DONE ===")
